In [ ]:
import random
from google.colab import output

In [ ]:
class Carta():

  valores = [str(i) for i in range(10)]
  colores = ["rojo", "amarillo", "verde", "azul"]
  valores_especiales = ["cancelar", "reverso", "+2"]
  comodines = ["+4", "cambio de color"]

  def __init__(self, valor, color):
    self.valor = valor
    self.color = color

  def __str__(self):
    return f'{self.valor} {self.color}'

  def __repr__(self):
    return str(self)

In [ ]:
class Mazo():

  def __init__(self):
    self.cartas = []

  def agregar_carta(self, carta):
    self.cartas.append(carta)

  def quitar_carta(self, idx):
    return self.cartas.pop(idx)

  def quitar_ultima_carta(self):
    return self.cartas.pop()

  def obtener_ultima_carta(self):
    return self.cartas[-1]

  def barajar(self):
    random.shuffle(self.cartas)

In [ ]:
class Jugador():

  def __init__(self, nombre):
    self.nombre = nombre
    self.mazo = Mazo()

In [ ]:
class Juego():

  def __init__(self, n_jugadores=4):

    self.mazo = self.__inicializar_mazo()
    self.jugadores = self.__crear_jugadores(n_jugadores)
    self.__repartir_cartas()

    self.mazo_descarte = Mazo()
    self.mazo_descarte.agregar_carta(self.mazo.quitar_ultima_carta())
    self.n_jugadores = n_jugadores
    self.turno = 0

    self.sentido = 1

  def obtener_movimientos_disponibles_jugador(self, jugador):

    movimientos_disponibles = []
    carta_descartada = self.mazo_descarte.obtener_ultima_carta()

    for idx, carta in enumerate(jugador.mazo.cartas):

      if( carta.valor == carta_descartada.valor
          or carta.color == carta_descartada.color
          or carta.valor in Carta.comodines):

        movimientos_disponibles.append(idx)

    return movimientos_disponibles

  def iniciar_partida(self):

    self.__main_loop()

  def poner_de_la_pila(self, jugador, n=1):
    for _ in range(n):
      jugador.mazo.agregar_carta(self.mazo.quitar_ultima_carta())

  def __main_loop(self):

    while True:

      curr_player = self.jugadores[self.turno%self.n_jugadores]
      print(f"Turno del jugador {curr_player.nombre}")
      print(f"Mazo del jugador {curr_player.mazo.cartas}")
      print(f"Carta de descarte: {self.mazo_descarte.obtener_ultima_carta()}")

      acciones_disponibles = self.obtener_movimientos_disponibles_jugador(curr_player)
      acciones_disponibles += ["Tomar de la pila"]

      print("Acciones disponibles")
      for i, accion in enumerate(acciones_disponibles[:-1]):
        print(f"{i}) {curr_player.mazo.cartas[accion]}")
      print(f"{len(acciones_disponibles)-1}) Tomar de la pila")

      accion_seleccionada = int(input("Seleccione una acción: "))

      # Si quiere tomar de la pila
      if accion_seleccionada == len(acciones_disponibles)-1:
        self.poner_de_la_pila(curr_player)
      # Si quiere jugar un carta
      else:
        carta = curr_player.mazo.quitar_carta(acciones_disponibles[accion_seleccionada])
        next_player = self.jugadores[(self.turno+1)%self.n_jugadores]
        # Si es un +2
        if carta.valor == "+2":
          self.poner_de_la_pila(next_player, n=2)
        # Si es un +4
        if carta.valor == "+4":
          self.poner_de_la_pila(next_player, n=4)
        # Si es cambio de color
        if carta.valor in ["cambio de color", "+4"]:
          color = input("Ingrese el nuevo color: ")
          carta.color = color
        # Si es cancelar turno
        if carta.valor in ["cancelar", "+2", "+4"]:
          self.turno += self.sentido
        # Si es reverso
        if carta.valor == "reverso":
          self.sentido *= -1
        self.mazo_descarte.agregar_carta(carta)

      self.turno += self.sentido

      output.clear()

  def __crear_jugadores(self, n_jugadores):

    jugadores = []

    assert n_jugadores > 1, "El número de jugadores debe ser mayor a 1"

    for i in range(n_jugadores):
      nombre = input(f"Ingrese el nombre del jugador {i+1}: ")
      jugador = Jugador(nombre)
      jugadores.append(jugador)

    return jugadores

  def __repartir_cartas(self):

    self.mazo.barajar()

    for jugador in self.jugadores:
      for _ in range(7):
        carta = self.mazo.quitar_ultima_carta()
        jugador.mazo.agregar_carta(carta)

  def __inicializar_mazo(self):

    # Inicializar mazo vacío
    mazo = Mazo()

    # Añadir dos cartas por combinacion de color-valor/especial
    opciones_valores = Carta.valores_especiales + Carta.valores[1:]
    for color in Carta.colores:
      for valor in opciones_valores:
        carta = Carta(valor, color)
        mazo.agregar_carta(carta)
        mazo.agregar_carta(carta)

    # Añadir cuatro cartas por cada comodín
    for comodin in Carta.comodines:
      for _ in range(4):
        carta = Carta(comodin, None)
        mazo.agregar_carta(carta)

    #Añadir los cuatro ceros
    for color in Carta.colores:
      carta = Carta(Carta.valores[0], color)
      mazo.agregar_carta(carta)

    return mazo

In [ ]:
juego = Juego()

Ingrese el nombre del jugador 1: A
Ingrese el nombre del jugador 2: B
Ingrese el nombre del jugador 3: C
Ingrese el nombre del jugador 4: D


In [ ]:
juego.iniciar_partida()

Turno del jugador C
Mazo del jugador [reverso rojo, 7 rojo, 2 azul, 5 azul, +4 None]
Carta de descarte: cambio de color azul
Acciones disponibles
0) 2 azul
1) 5 azul
2) +4 None
3) Tomar de la pila


KeyboardInterrupt: Interrupted by user